<a href="https://colab.research.google.com/github/bhushan1729/olfaction-inspired-ner/blob/main/notebooks/universal_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Olfactory NER Experiments

This notebook allows running Olfaction-inspired NER experiments across different datasets (CoNLL-2003, WikiANN, etc.) and languages using a unified framework.

**Architecture:**
- **Receptor Layer:** Specialized feature detectors inspired by olfactory receptors.
- **Glomerular Layer:** Aggregates receptor activations.
- **BiLSTM-CRF:** Standard backbone for NER.

In [ ]:
# @title 1. Configuration & Setup
# ==============================================================================

# ------------------------------------------------------------------------------
# USER INPUTS - CHANGE THESE
# ------------------------------------------------------------------------------
MOUNT_DRIVE = True         # Set to True to save results to Google Drive

# Select experiments to run
# ------------------------------------------------------------------------------

import os
import sys
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Mount Drive if requested
if MOUNT_DRIVE and IN_COLAB:
    try:
        # force_remount=True fixes 'Mountpoint must not already contain files' error
        drive.mount('/content/drive', force_remount=True)
        BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral'
    except Exception as e:
        print(f"Warning: Failed to mount Drive ({e}). Using local storage.")
        BASE_SAVE_DIR = './results'
elif MOUNT_DRIVE and not IN_COLAB:
    print("Info: MOUNT_DRIVE=True but running locally. Using local storage.")
    BASE_SAVE_DIR = './results'
else:
    BASE_SAVE_DIR = './results'

print(f"Results will be saved to: {BASE_SAVE_DIR}")

# Git Setup
# Extract ZIP format from Drive (Workaround for GitHub limits)
import zipfile
if IN_COLAB:
    zip_path = '/content/drive/My Drive/olfaction-inspired-ner-mitral.zip'
    extract_dir = '/content/olfaction-inspired-ner'
    if not os.path.exists(extract_dir + '/olfaction-inspired-ner'):
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
    os.chdir(extract_dir + '/olfaction-inspired-ner')

# Check if we are already in the project root (e.g. local execution)
if os.path.exists('src/train_universal.py') and os.path.exists('config/mitral_config.yaml'):
    print("Already in project root.")
elif os.path.exists('olfaction-inspired-ner'):
    !git pull
else:
    # Clone using token
    print("Please provide your GitHub Personal Access Token to clone the private repository.")

# Install dependencies
!pip install datasets pyyaml seqeval

# Add src to path
sys.path.append(os.getcwd())

In [ ]:
    # Get your token securely (won't be visible in output)

    # Clone using token


In [ ]:
# Patch src/train_universal.py to remove 'verbose=True'
fn = 'src/train_universal.py'
with open(fn, 'r') as f:
    content = f.read()

if 'verbose=True' in content:
    print(f"Patching {fn}...")
    content = content.replace(', verbose=True', '')
    with open(fn, 'w') as f:
        f.write(content)
    print("Success: Removed verbose=True argument.")
else:
    print(f"{fn} already patched or 'verbose=True' not found.")

In [ ]:
# @title Run ALL Experiments (All Languages)
import os
import yaml

CONFIG_PATH = 'config/mitral_config.yaml'
BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

# 1. Load Config to get all experiments
with open(CONFIG_PATH, 'r') as f:
    full_config = yaml.safe_load(f)

# Get all experiment keys
all_experiments = list(full_config['experiments'].keys())

# Define Datasets
DATASETS = ['conll_en', 'wikiann_hi', 'wikiann_mr', 'wikiann_ta', 'wikiann_bn', 'wikiann_te']

print(f"Running on Datasets: {DATASETS}")
print(f"Running Experiments: {all_experiments}")

for dataset_key in DATASETS:
    print(f"\n{'#'*80}")
    print(f"PROCESSING DATASET: {dataset_key}")
    print(f"{'#'*80}")

    for exp in all_experiments:
        print(f"\n>>> Running Experiment: {exp} on {dataset_key} <<<")

        !python src/train_universal.py \
            --config {CONFIG_PATH} \
            --dataset_key {dataset_key} \
            --experiment {exp} \
            --save_dir "{BASE_SAVE_DIR}"

        print(f"✓ Completed {exp} on {dataset_key}")

In [ ]:
# @title Fix Config & Run WikiANN Bangla/Telugu
import os
import yaml

CONFIG_PATH = 'config/mitral_config.yaml'
BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

print("Fixing Configuration File...")

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r') as f:
        config = yaml.safe_load(f)

    # Define the new datasets
    new_datasets = {
        'wikiann_bn': {
            'dataset': "wikiann",
            'language': "bn",
            'glove_path': None,
            'use_pretrained_embeddings': False
        },
        'wikiann_te': {
            'dataset': "wikiann",
            'language': "te",
            'glove_path': None,
            'use_pretrained_embeddings': False
        }
    }

    # 1. Clean up: Remove if they were accidentally added to 'experiments'
    if 'experiments' in config:
        for key in ['wikiann_bn', 'wikiann_te']:
            if key in config['experiments']:
                print(f"Removing {key} from experiments section...")
                del config['experiments'][key]

    # 2. Add to 'datasets' section
    if 'datasets' not in config:
        config['datasets'] = {}

    for key, val in new_datasets.items():
        config['datasets'][key] = val
        print(f"Added {key} to datasets section.")

    # 3. Save corrected config
    with open(CONFIG_PATH, 'w') as f:
        yaml.dump(config, f, sort_keys=False, default_flow_style=False)
    print("✓ Configuration file corrected and saved.")

else:
    print(f"Error: {CONFIG_PATH} not found!")

# ------------------------------------------------------------------------------
# Run Experiments
# ------------------------------------------------------------------------------
NEW_DATASETS = ['wikiann_bn', 'wikiann_te']
ALL_EXPERIMENTS = list(config['experiments'].keys())

print(f"\nRunning on Datasets: {NEW_DATASETS}")
print(f"Running Experiments: {ALL_EXPERIMENTS}")

for dataset_key in NEW_DATASETS:
    print(f"\n{'#'*80}")
    print(f"PROCESSING DATASET: {dataset_key}")
    print(f"{'#'*80}")

    for exp in ALL_EXPERIMENTS:
        print(f"\n>>> Running Experiment: {exp} on {dataset_key} <<<")

        !python src/train_universal.py \
            --config {CONFIG_PATH} \
            --dataset_key {dataset_key} \
            --experiment {exp} \
            --save_dir "{BASE_SAVE_DIR}"

        print(f"✓ Completed {exp} on {dataset_key}")

In [ ]:
# @title 3. Generate Visualizations & Metrics (Corrected)
# ==============================================================================
import torch
import json
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys

# Ensure src is in path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.model.olfactory_ner import create_olfactory_ner
from src.data.unified_loader import get_dataset
from src.analysis.visualize import analyze_receptor_activations

# Setup paths (same as before)
if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Scanning results in: {BASE_SAVE_DIR}")

# Scan directory for datasets
if not os.path.exists(BASE_SAVE_DIR):
    print("Results directory not found!")
else:
    # Get all dataset folders
    datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]

    for dataset_dir_name in datasets_found: # e.g., 'conll2003' or 'wikiann'
        d_path = os.path.join(BASE_SAVE_DIR, dataset_dir_name)

        # Get Language folders
        languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]

        for lang in languages: # e.g., 'default' or 'hi'
            print(f"\n{'='*80}")
            print(f"Processing: {dataset_dir_name} - {lang}")
            print(f"{'='*80}")

            full_results_dir = os.path.join(d_path, lang)
            viz_dir = os.path.join(full_results_dir, 'visualizations')
            os.makedirs(viz_dir, exist_ok=True)

            # Identify experiments in this folder
            experiments_found = [e for e in os.listdir(full_results_dir)
                                if os.path.isdir(os.path.join(full_results_dir, e)) and e != 'visualizations']

            if not experiments_found:
                print("No experiments found.")
                continue

            # Load Data specific to this dataset/language
            # We need to map the directory structure back to dataset keys for get_dataset
            # This is tricky if names differ, but let's try standard names.
            # Usually dataset_name passed to get_dataset is 'conll2003' or 'wikiann'

            print("Loading test data...")
            try:
                _, _, test_loader, vocab_info = get_dataset(
                    dataset_name=dataset_dir_name,
                    language=lang if lang != 'default' else 'en', # guess 'en' for default
                    cache_dir='./data'
                )
            except Exception as e:
                print(f"Warning: Could not load data for {dataset_dir_name}/{lang}. Skipping heatmaps. Error: {e}")
                test_loader = None

            # ------------------------------------------------------------------
            # Generate Heatmaps
            # ------------------------------------------------------------------
            if test_loader:
                print("Generating Heatmaps...")
                for exp in experiments_found:
                    # Skip if baseline (no receptors)
                    if 'baseline' in exp: continue

                    model_path = os.path.join(full_results_dir, exp, 'best_model.pt')
                    results_path = os.path.join(full_results_dir, exp, 'results.json')

                    if not os.path.exists(model_path): continue

                    try:
                        # Load config
                        with open(results_path, 'r') as f:
                            res = json.load(f)
                        model_config = res['config']

                        if not model_config.get('use_receptors', True): continue

                        # Load Model
                        model = create_olfactory_ner(len(vocab_info['word2idx']), len(vocab_info['label2idx']), model_config)
                        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
                        model.load_state_dict(checkpoint['model_state_dict'])
                        model = model.to(device)

                        save_path_exp = os.path.join(viz_dir, exp)
                        analyze_receptor_activations(model, test_loader, vocab_info, device, save_dir=save_path_exp, experiment_name=exp)
                        print(f"✓ Heatmaps generated for {exp}")

                    except Exception as e:
                        print(f"Error processing {exp}: {e}")

            # ------------------------------------------------------------------
            # Aggregate Metrics for this Language
            # ------------------------------------------------------------------
            metrics = []
            for exp in experiments_found:
                res_path = os.path.join(full_results_dir, exp, 'results.json')
                if os.path.exists(res_path):
                    try:
                        with open(res_path, 'r') as f:
                            data = json.load(f)
                        test_res = data['test']
                        metrics.append({
                            'Experiment': exp,
                            'F1': test_res['f1'],
                            'Precision': test_res['precision'],
                            'Recall': test_res['recall']
                        })
                    except:
                        pass

            if metrics:
                df = pd.DataFrame(metrics)
                print(f"\nResults Summary ({lang}):")
                print(df.sort_values(by='F1', ascending=False).to_string(index=False))
                df.to_csv(os.path.join(viz_dir, 'summary_metrics.csv'), index=False)

In [ ]:
# @title 4. Save Olfactory Metrics to Results
from src.analysis.visualize import analyze_receptor_activations
from src.model.olfactory_ner import create_olfactory_ner
from src.data.unified_loader import get_dataset
import torch
import json
import os
import sys

# Ensure src is in path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Updating results in: {BASE_SAVE_DIR}")

# Scan for datasets
datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]

for dataset_name in datasets_found:
    d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
    languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]

    for lang in languages:
        full_results_dir = os.path.join(d_path, lang)
        print(f"\nProcessing {dataset_name}/{lang}...")

        # Load Data once per dataset/language
        try:
            _, _, test_loader, vocab_info = get_dataset(
                dataset_name=dataset_name,
                language=lang if lang != 'default' else 'en',
                cache_dir='./data'
            )
        except Exception as e:
            print(f"Skipping data load for {dataset_name} ({e})")
            continue

        experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e))]

        for exp in experiments:
            if 'baseline' in exp: continue

            res_path = os.path.join(full_results_dir, exp, 'results.json')
            model_path = os.path.join(full_results_dir, exp, 'best_model.pt')

            if os.path.exists(res_path) and os.path.exists(model_path):
                # Load existing results
                with open(res_path, 'r') as f:
                    results = json.load(f)

                # Check if metrics already exist to save time (optional, remove check to force update)
                # if 'olfactory_metrics' in results: continue

                config = results['config']
                if not config.get('use_receptors', True): continue

                print(f"  Analyzing {exp}...")

                # Load Model
                try:
                    model = create_olfactory_ner(len(vocab_info['word2idx']), len(vocab_info['label2idx']), config)
                    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
                    model.load_state_dict(checkpoint['model_state_dict'])
                    model = model.to(device)

                    # Run Analysis (suppress plots by saving to temp or just ignoring)
                    # analyze_receptor_activations returns a dict with metrics
                    # We pass a dummy save_dir because the function tries to save plots
                    dummy_dir = os.path.join(full_results_dir, exp, 'temp_viz')
                    os.makedirs(dummy_dir, exist_ok=True)

                    analysis_stats = analyze_receptor_activations(
                        model, test_loader, vocab_info, device,
                        save_dir=dummy_dir, experiment_name=exp
                    )

                    # Extract Metrics
                    olfactory_metrics = {
                        'sparsity': analysis_stats.get('sparsity', 0),
                        'avg_activation': analysis_stats.get('avg_activation', 0),
                        'avg_rsi': analysis_stats.get('avg_rsi', 0)
                    }

                    # Update Results
                    results['olfactory_metrics'] = olfactory_metrics

                    # Save back
                    with open(res_path, 'w') as f:
                        json.dump(results, f, indent=2)

                    print(f"    -> Saved Sparsity: {olfactory_metrics['sparsity']:.2%}, RSI: {olfactory_metrics['avg_rsi']:.4f}")

                except Exception as e:
                    print(f"    -> Error: {e}")

print("\nDONE. All results.json files updated.")

In [ ]:
# @title 4. Compare Results (F1 Score Bar Plots for All Datasets)
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Setup paths
if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

print(f"Generating plots from: {BASE_SAVE_DIR}")

if not os.path.exists(BASE_SAVE_DIR):
    print("Results directory not found.")
else:
    # Scan for datasets
    datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]

    for dataset_name in datasets_found:
        d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
        languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]

        for lang in languages:
            full_results_dir = os.path.join(d_path, lang)
            viz_dir = os.path.join(full_results_dir, 'visualizations')
            os.makedirs(viz_dir, exist_ok=True)

            # Collect metrics for this dataset/language
            metrics = []
            experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e)) and e != 'visualizations']

            for exp in experiments:
                res_path = os.path.join(full_results_dir, exp, 'results.json')
                if os.path.exists(res_path):
                    try:
                        with open(res_path, 'r') as f:
                            data = json.load(f)
                        metrics.append({
                            'Experiment': exp,
                            'F1': data['test']['f1']
                        })
                    except:
                        pass

            # Plot
            if metrics:
                df = pd.DataFrame(metrics)
                df = df.sort_values(by='F1', ascending=False)

                plt.figure(figsize=(10, 6))
                sns.barplot(x='F1', y='Experiment', data=df, palette='viridis')
                plt.title(f'F1 Score Comparison - {dataset_name} ({lang})')
                plt.xlim(0, 1.0)
                plt.grid(axis='x', alpha=0.3)
                plt.tight_layout()

                # Save and Show
                save_path = os.path.join(viz_dir, 'f1_comparison.png')
                plt.savefig(save_path, dpi=300)
                plt.show()
                print(f"✓ Generated plot for {dataset_name}/{lang}")
                plt.close()
            else:
                print(f"No results found for {dataset_name}/{lang}")

In [ ]:
# @title 4b. Plot Per-Entity F1 Scores
# ==============================================================================
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Setup paths (ensure BASE_SAVE_DIR is correct)
if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

print(f"Generating Per-Entity plots from: {BASE_SAVE_DIR}")

if not os.path.exists(BASE_SAVE_DIR):
    print("Results directory not found.")
else:
    # Scan for datasets
    datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]

    for dataset_name in datasets_found:
        d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
        languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]

        for lang in languages:
            full_results_dir = os.path.join(d_path, lang)
            viz_dir = os.path.join(full_results_dir, 'visualizations')
            os.makedirs(viz_dir, exist_ok=True)

            # Collect Per-Entity metrics
            data_records = []
            experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e)) and e != 'visualizations']

            for exp in experiments:
                res_path = os.path.join(full_results_dir, exp, 'results.json')
                if os.path.exists(res_path):
                    try:
                        with open(res_path, 'r') as f:
                            res = json.load(f)

                        test_metrics = res['test']
                        per_entity = test_metrics.get('per_entity', {})

                        for entity, f1 in per_entity.items():
                            data_records.append({
                                'Experiment': exp,
                                'Entity Type': entity,
                                'F1 Score': f1
                            })
                    except Exception as e:
                        print(f"Skipping {exp}: {e}")

            # Plot
            if data_records:
                df_entity = pd.DataFrame(data_records)

                plt.figure(figsize=(14, 8))

                # Grouped Bar Plot
                chart = sns.barplot(
                    data=df_entity,
                    x='Entity Type',
                    y='F1 Score',
                    hue='Experiment',
                    palette='tab10',
                    edgecolor='black'
                )

                plt.title(f'Per-Entity F1 Score Comparison ({dataset_name} - {lang})', fontsize=16, fontweight='bold')
                plt.ylim(0, 1.05)
                plt.grid(axis='y', alpha=0.3)

                # Legend placement outside
                plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0.)
                plt.tight_layout()

                # Save and Show
                save_path = os.path.join(viz_dir, 'per_entity_f1_comparison.png')
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"✓ Generated Per-Entity plot for {dataset_name}/{lang}")
                plt.close()
            else:
                print(f"No per-entity data found for {dataset_name}/{lang}")

In [ ]:
# @title 4c. Plot Precision vs Recall (Bubble Chart)
# ==============================================================================
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Setup paths
if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

print(f"Generating Bubble Plots from: {BASE_SAVE_DIR}")

if not os.path.exists(BASE_SAVE_DIR):
    print("Results directory not found.")
else:
    # Scan for datasets
    datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]

    for dataset_name in datasets_found:
        d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
        languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]

        for lang in languages:
            full_results_dir = os.path.join(d_path, lang)
            viz_dir = os.path.join(full_results_dir, 'visualizations')
            os.makedirs(viz_dir, exist_ok=True)

            # Collect metrics
            metrics = []
            experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e)) and e != 'visualizations']

            for exp in experiments:
                res_path = os.path.join(full_results_dir, exp, 'results.json')
                if os.path.exists(res_path):
                    try:
                        with open(res_path, 'r') as f:
                            data = json.load(f)
                        metrics.append({
                            'Experiment': exp,
                            'F1': data['test']['f1'],
                            'Precision': data['test']['precision'],
                            'Recall': data['test']['recall']
                        })
                    except:
                        pass

            # Plot
            if metrics:
                df = pd.DataFrame(metrics)

                plt.figure(figsize=(12, 10))

                # Bubble Plot: X=Recall, Y=Precision, Size=F1
                scatter = sns.scatterplot(
                    data=df,
                    x='Recall',
                    y='Precision',
                    size='F1',
                    sizes=(400, 2000), # Adjust bubble sizes
                    hue='Experiment',
                    palette='tab10',
                    alpha=0.7,
                    edgecolor='black',
                    legend=False
                )

                # Annotate points
                for i, row in df.iterrows():
                    plt.text(
                        row['Recall'],
                        row['Precision'],
                        f"{row['Experiment']}",
                        fontsize=10,
                        ha='center',
                        va='center',
                        fontweight='bold'
                    )

                plt.title(f'Precision vs Recall (Bubble Size = F1 Score) - {dataset_name}/{lang}', fontsize=16, fontweight='bold')
                plt.xlabel('Recall', fontsize=12, fontweight='bold')
                plt.ylabel('Precision', fontsize=12, fontweight='bold')
                plt.grid(True, alpha=0.3)
                plt.tight_layout()

                # Save and Show
                save_path = os.path.join(viz_dir, 'precision_recall_bubble.png')
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"✓ Generated Bubble plot for {dataset_name}/{lang}")
                plt.close()
            else:
                print(f"No results found for {dataset_name}/{lang}")

In [ ]:
# @title 4d. Comprehensive Metrics Table (Dataset-wide)
# ==============================================================================
import os
import json
import pandas as pd

# Setup paths
if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner_mitral' if os.path.exists('/content/drive') else './results'

print(f"Generating Tables from: {BASE_SAVE_DIR}")

if not os.path.exists(BASE_SAVE_DIR):
    print("Results directory not found.")
else:
    # Scan for datasets
    datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]

    for dataset_name in datasets_found:
        d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
        languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]

        for lang in languages:
            full_results_dir = os.path.join(d_path, lang)
            viz_dir = os.path.join(full_results_dir, 'visualizations')
            os.makedirs(viz_dir, exist_ok=True)

            print(f"\n{'#'*100}")
            print(f"DATASET: {dataset_name} | LANGUAGE: {lang}")
            print(f"{'#'*100}")

            # Collect metrics
            metrics_data = []
            experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e)) and e != 'visualizations']

            for exp_name in experiments:
                res_path = os.path.join(full_results_dir, exp_name, 'results.json')
                if os.path.exists(res_path):
                    try:
                        with open(res_path, 'r') as f:
                            data = json.load(f)

                        test_metrics = data.get('test', {})
                        per_entity = test_metrics.get('per_entity', {})

                        # Extract metrics matching your desired format
                        # Note: keys for per_entity depend on the dataset (LOC/PER/ORG vs others)
                        # We try to get common ones, or default to 0

                        row = {
                            'Experiment': exp_name,
                            'F1': test_metrics.get('f1', 0),
                            'Precision': test_metrics.get('precision', 0),
                            'Recall': test_metrics.get('recall', 0),
                            # Try standard entities, use .get() to be safe
                            'LOC_F1': per_entity.get('LOC', 0),
                            'MISC_F1': per_entity.get('MISC', 0),
                            'ORG_F1': per_entity.get('ORG', 0),
                            'PER_F1': per_entity.get('PER', 0),
                            # Averages might be stored directly or inside per_entity depending on implementation
                            # If not present, we use Overall F1 as Micro Avg proxy
                            'Micro_Avg': per_entity.get('micro avg', test_metrics.get('f1', 0)),
                            'Macro_Avg': per_entity.get('macro avg', 0),
                            'Weighted_Avg': per_entity.get('weighted avg', 0)
                            # If your evaluation code calculates macro/weighted, fetch it here.
                            # Standard seqeval output often puts averages in the report dict,
                            # but simpler implementations might not save them explicitly.
                            # We'll check if they exist in per_entity (output of classification_report(output_dict=True))
                        }

                        # Add any other entities dynamically if they exist and aren't standard
                        # (Optional: uncomment to see all dynamic keys)
                        # for k, v in per_entity.items():
                        #     if k not in ['LOC', 'MISC', 'ORG', 'PER']:
                        #         row[f'{k}_F1'] = v

                        metrics_data.append(row)
                    except Exception as e:
                        print(f"Error reading {exp_name}: {e}")

            # Create DataFrame
            if metrics_data:
                metrics_df = pd.DataFrame(metrics_data)

                # Sort by F1 score descending
                metrics_df = metrics_df.sort_values('F1', ascending=False)

                # Display
                print(metrics_df.round(4).to_string(index=False))

                # Save
                save_path = os.path.join(viz_dir, 'comprehensive_metrics_table.csv')
                metrics_df.to_csv(save_path, index=False)
                print(f"\n✓ Table saved to: {save_path}")
            else:
                print("No experiment data found.")
            print("\n")

In [ ]:
# done